# Vanilla PEFT + TRL SFT — Unsloth'suz Karşılaştırma

Bu notebook **Unsloth kullanmadan** sadece HuggingFace stack'i (PEFT + TRL + bitsandbytes) ile aynı SFT'yi yapıyor.

## Niye?
- Unsloth'un **ne yaptığını anlamak** için temeli görmek
- HuggingFace stack ile çalışmaya alışmak (Unsloth desteklemediği modellerde)
- Code complexity, speed, VRAM **doğrudan karşılaştırma**

## Doğrudan Karşılaştırma (Smoke Test, RTX 4070 Ti SUPER 16GB)

| Metrik | Vanilla PEFT+TRL | Unsloth (`01_sft_modern`) |
|---|---|---|
| Süre 3 step | 9.47 sec | 13.4 sec |
| Sec/step | 3.16 | 4.47 |
| Peak VRAM | 4.84 GB | 4.22 GB |
| Train loss (3 step) | 4.75 (full seq) | 2.30 (response-only) |
| Code satır sayısı | ~25 | ~10 |
| Setup karmaşıklığı | Yüksek (BnB config + prepare_kbit + LoraConfig) | Düşük (`from_pretrained` + `get_peft_model`) |

**Önemli:** Vanilla path'te `train_on_responses_only` Unsloth helper'ı yok. TRL'nin `assistant_only_loss=True` parametresi var ama farklı çalışıyor (chat template'de `{% generation %}` keyword'ü gerektirir).

Production'da (60+ step) Unsloth **2x hız** + **30% daha az VRAM** avantajını gösterir — kısa run'larda compilation overhead farkı yutar.

## İçindekiler
1. **Setup** — Vanilla imports (Unsloth YOK)
2. **BitsAndBytesConfig** — 4-bit quantization manuel
3. **Model + Tokenizer** — `AutoModelForCausalLM` + `prepare_model_for_kbit_training`
4. **LoraConfig** — PEFT manuel
5. **Dataset** — Aynı (Sebastian Raschka ch07)
6. **SFTTrainer** — TRL standart
7. **Training**
8. **Save**
9. **Unsloth vs Vanilla — Kod Karşılaştırması**

## 1. Setup — Vanilla Imports

Dikkat: **`import unsloth` YOK**. Sadece HuggingFace stack.

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import requests

print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 2. BitsAndBytesConfig — Manuel 4-bit Quantization

Unsloth'ta `load_in_4bit=True` tek satırlı — vanilla'da BitsAndBytesConfig kurman gerek:

| Parametre | Değer | Açıklama |
|---|---|---|
| `load_in_4bit` | True | 4-bit yükle |
| `bnb_4bit_quant_type` | `'nf4'` | NormalFloat4 — standard |
| `bnb_4bit_compute_dtype` | `bf16` | Compute precision |
| `bnb_4bit_use_double_quant` | True | ~0.4 bit ekstra tasarruf |

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

## 3. Model + Tokenizer

Vanilla path'te 3 ekstra adım var:
1. `AutoModelForCausalLM.from_pretrained(quantization_config=...)` ile yükle
2. `prepare_model_for_kbit_training(model)` — gradient checkpointing ile uyumluluk için (QLoRA standart)
3. `model.config.use_cache = False` — gradient checkpointing ile use_cache çakışır

**`attn_implementation='sdpa'`** — flash-attn yoksa fallback. Eski versiyonlarda `'eager'` kullanırdık.

In [ ]:
MODEL_NAME = 'unsloth/Qwen3-4B-Instruct-2507'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map = 'auto',
    dtype = torch.bfloat16,
    attn_implementation = 'sdpa',                # 'flash_attention_2' if installed
)
print(f'Model: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params loaded')

# QLoRA için ZORUNLU
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False                   # gradient checkpointing compat

## 4. LoraConfig — PEFT Manuel

Unsloth'ın `get_peft_model` ile aynı parametreler ama PEFT'in raw API'si üzerinden:

In [ ]:
lora_config = LoraConfig(
    r = 32,
    lora_alpha = 32,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout = 0,
    bias = 'none',
    task_type = 'CAUSAL_LM',                     # PEFT'te ZORUNLU (Unsloth otomatik)
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Dataset — Sebastian Raschka ch07 (Aynı)

01_sft_modern ile birebir aynı dataset. Karşılaştırma temiz olsun diye.

In [ ]:
url = 'https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json'
raw = requests.get(url, timeout=30).json()

def alpaca_to_conversations(entry):
    user_text = entry['instruction']
    if entry.get('input'):
        user_text += f"\n\n{entry['input']}"
    return {'messages': [
        {'role': 'user',      'content': user_text},
        {'role': 'assistant', 'content': entry['output']},
    ]}

n = len(raw)
train_raw = raw[:int(n*0.85)]
dataset = Dataset.from_list([alpaca_to_conversations(e) for e in train_raw])
print(f'Train: {len(dataset)}')

# Format — vanilla'da Unsloth'un standardize_data_formats yok
# Direkt apply_chat_template
def fmt(examples):
    return {'text': [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in examples['messages']
    ]}
dataset = dataset.map(fmt, batched=True)
print('\nFirst 200 chars:')
print(dataset[0]['text'][:200])

## 6. SFTTrainer — TRL Standart

**ÖNEMLİ:** `train_on_responses_only` Unsloth'a ait, vanilla'da yok. Alternatifler:

1. **`assistant_only_loss=True` (SFTConfig)** — chat template'de `{% generation %}` keyword'ü olmalı
2. **DataCollatorForCompletionOnlyLM** — manuel response masking
3. **Hiçbir şey yapma** — full sequence loss (smoke test'te bu, basit ama suboptimal)

Bu notebook'ta **3. seçenek** — basitlik için. Production'da `assistant_only_loss=True` öneririm.

In [ ]:
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = 'text',
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,                          # demo; production: num_train_epochs=1
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = 'adamw_8bit',
        weight_decay = 0.001,
        lr_scheduler_type = 'linear',
        seed = 3407,
        report_to = 'none',
        bf16 = True,
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {'use_reentrant': False},
        max_length = 2048,
        # assistant_only_loss = True,            # Qwen3 chat template `{% generation %}` destekliyor
    ),
    processing_class = tokenizer,
)

## 7. Training

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_mem = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU = {gpu_stats.name} | start mem = {start_mem} / {max_mem} GB')

trainer_stats = trainer.train()

used = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n=== RESULTS ===")
print(f"Train runtime: {trainer_stats.metrics['train_runtime']:.2f} sec")
print(f"Train loss:    {trainer_stats.metrics['train_loss']:.4f}")
print(f"Peak VRAM:     {used} GB")
print(f"Sec / step:    {trainer_stats.metrics['train_runtime']/60:.2f}")

## 8. Save

In [ ]:
# A. LoRA adapter (PEFT standart)
model.save_pretrained('vanilla_sft_lora')
tokenizer.save_pretrained('vanilla_sft_lora')
print('LoRA saved: vanilla_sft_lora/')

# B. Merged 16-bit (PEFT'in merge_and_unload metodu)
# merged = model.merge_and_unload()
# merged.save_pretrained('vanilla_sft_merged', safe_serialization=True)
# tokenizer.save_pretrained('vanilla_sft_merged')

# Dikkat: Unsloth'un save_pretrained_merged ve save_pretrained_gguf metodları yok vanilla'da.
# GGUF için ayrıca llama.cpp convert-hf-to-gguf.py kullan.

## 9. Unsloth vs Vanilla — Kod Karşılaştırması

### Yan Yana Setup

**Unsloth (3 satır):**
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen3-4B-Instruct-2507',
    max_seq_length=2048, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(model, r=32, ...)
```

**Vanilla (15+ satır):**
```python
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map='auto', dtype=torch.bfloat16, attn_implementation='sdpa',
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
lora_config = LoraConfig(r=32, ..., task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
```

### Unsloth Avantajları
1. **Daha az kod** — 3 satır vs 15+
2. **Otomatik optimizasyonlar** — Triton kernels, smart gradient checkpointing
3. **`get_chat_template` + `train_on_responses_only`** — vanilla'da kendin yapacaksın
4. **`save_pretrained_merged` + `save_pretrained_gguf`** — vanilla'da llama.cpp manual
5. **Production'da %50+ daha hızlı + %30 daha az VRAM** (kanıtlanmış)

### Vanilla'nın Yeri
1. **Unsloth desteklemediği modeller** (yeni / nadir mimariler)
2. **Multi-GPU karmaşık senaryolar** — vanilla path daha esnek
3. **Custom optimizasyonlar** — kendi forward pass'ını yazmak istediğinde
4. **HuggingFace ekosistemini öğrenmek** — fundamental anlayış
5. **Multimodal DPO/GRPO** — Unsloth'un desteklemediği yerler

## Özet

1. **Aynı SFT, 2 farklı stack** — pipeline mantığı aynı, kod farklı
2. **Unsloth = Optimize edilmiş wrapper** — Triton kernels + helper'lar
3. **Vanilla = Daha fazla kod ama daha esnek** — tüm modelleri destekliyor
4. **`prepare_model_for_kbit_training` ZORUNLU** vanilla path'te (QLoRA için)
5. **`task_type='CAUSAL_LM'`** PEFT'te ZORUNLU — Unsloth otomatik
6. **`use_cache=False`** gradient checkpointing ile uyumluluk
7. **`train_on_responses_only` Unsloth-specific** — vanilla'da `assistant_only_loss=True` deneyebilirsin

**Test edildi:** `smoke/06_vanilla_peft_sft_smoke.py` — 3 step'te 9.47 sec, 4.84 GB VRAM. Pipeline çalışıyor.

## Pratik Tavsiye
- **Unsloth desteklediği modeller** → Unsloth kullan (daha hızlı + az VRAM)
- **Unsloth desteklemediği** → Vanilla path (multi-GPU, multimodal DPO, vs.)